# Cell Tokenization

- Creator: Sebastian Birk (sebastian.birk@helmholtz-munich.de).
- Date of Creation: 25.06.2024
- Date of Last Modification: 02.07.2024

## 1. Setup

### 1.1 Import Libraries

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
import os
import pickle
import random
import warnings
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import scgpt as scg
import seaborn as sns
import squidpy as sq
import torch
from datasets import load_from_disk
from scipy.stats import spearmanr
from scgpt.model import TransformerModel
from scgpt.tokenizer.gene_tokenizer import GeneVocab
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
from skmisc.loess import loess

from nichejepa import tokenizers
from nichejepa import utils

### 1.2 Run Notebook Setup

In [ ]:
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

### 1.3 Configure Paths and Create Directories

In [ ]:
# Define paths
sc_data_folder_path = "../../datasets/sc_data" # spatially resolved transcriptomics data
sc_data_bronze_folder_path = f"{sc_data_folder_path}/bronze"
sc_data_silver_folder_path = f"{sc_data_folder_path}/silver"
sc_data_gold_folder_path = f"{sc_data_folder_path}/gold"
tokenizer = "cell_tokenizer"
tokenizer_folder_path = f"../datasets/tokenizer/{tokenizer}"
results_folder_path = "../../datasets/results"
geneformer_token_dictionary_file = f"../datasets/tokenizer/geneformer_token_dictionary.pkl"
# dataset_cache_file = "../datasets/scgpt_gene_embeddings"
scgpt_model_dir = Path("../../scgpt/pretrained_model")
scgpt_model_config_file = scgpt_model_dir / "args.json"
scgpt_model_file = scgpt_model_dir / "best_model.pt"
scgpt_vocab_file = scgpt_model_dir / "vocab.json"
scgpt_metadata_file = Path("../../scgpt/metadata/gene_info.csv")

## 2. Tokenization

In [ ]:
# Define dataset
dataset_name = "snrnaseq_human_ms"

In [ ]:
add_cls_token = False
separate_cell_and_neighborhood_tokens = False

if dataset_name == "snrnaseq_human_ms":
    n_batches = 1
    species = "homo_sapiens"
    norm_factors = None
    custom_attr_name_dict = {"cell_types": "cell_types"}
    label_cats = ["cell_types"]

os.makedirs(f"{tokenizer_folder_path}/{dataset_name}", exist_ok=True)

# Set dataset-specific paths and parameters
cell_gene_means_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/cell_gene_means_dictionary.pkl"
cell_gene_nzmedians_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/cell_gene_nzmedians_dictionary.pkl"
cell_gene_logmeans_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/cell_gene_logmeans_dictionary.pkl"
neighborhood_gene_means_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/neighborhood_gene_means_dictionary.pkl"
neighborhood_gene_nzmedians_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/neighborhood_gene_nzmedians_dictionary.pkl"
neighborhood_gene_logmeans_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/neighborhood_gene_logmeans_dictionary.pkl"
nichejepa_token_dictionary_file = f"{tokenizer_folder_path}/{dataset_name}/token_dictionary" \
                                  f"{'_cls' if add_cls_token else '_no_cls'}.pkl"
token_dictionary_file = f"../datasets/tokenizer/token_dictionary_{species}.pkl"
input_directory = f"{sc_data_silver_folder_path}/{dataset_name}"
output_directory = f"{sc_data_gold_folder_path}/{tokenizer}/{dataset_name}"

### 2.1 Create Token Dictionary

In [ ]:
if species == "mus_musculus":
    # Extract Ensembl IDs of protein coding and miRNA mouse genes
    ensembl = EnsemblRelease(111, species=species)
    ensembl.download()
    ensembl.index()
    all_genes = ensembl.genes()
    protein_coding_genes = [gene for gene in all_genes if gene.biotype == "protein_coding"]
    mirna_genes = [gene for gene in all_genes if gene.biotype == "miRNA"]
    all_relevant_genes = protein_coding_genes + mirna_genes
    gene_ids = [gene.gene_id for gene in all_relevant_genes]
    
    # Create token dictionary for mouse
    token_dict = {'<pad>': 0, '<mask>': 1}
    token_dict.update({gene_id: idx + 2 for idx, gene_id in enumerate(gene_ids)})
    with open(token_dictionary_file, "wb") as f:
        pickle.dump(token_dict, f)
elif species == "homo_sapiens":
    # Load geneformer token dict to get protein-coding and miRNA genes for human
    with open(geneformer_token_dictionary_file, "rb") as f:
        token_dict = pickle.load(f)
    with open(token_dictionary_file, "wb") as f:
            pickle.dump(token_dict, f)

In [ ]:
# Load data
if n_batches == 1:
    adata = sc.read_h5ad(f"{sc_data_silver_folder_path}/{dataset_name}/{dataset_name}.h5ad")
else:
    adata = sc.read_h5ad(f"{sc_data_silver_folder_path}/{dataset_name}/{dataset_name}_batch1.h5ad")

In [ ]:
### Create token dict for nichejepa ###
with open(token_dictionary_file, "rb") as f:
    token_dict = pickle.load(f)

# Add cls token
if add_cls_token:
    def insert_at_index(d, index, key, value):
        items = list(d.items())
        items.insert(index, (key, value))
        new_d = dict(items)
        return new_d
    token_dict = insert_at_index(token_dict, 2, "<cls>", 2)

# Remove genes not in dataset
gene_list = adata.var["ensembl_id"].tolist()
token_dict_copy = token_dict.copy()
for token in token_dict_copy.keys():
    if "ENS" in token and token not in gene_list:
        del(token_dict[token])
del(token_dict_copy)

# Separate gene tokens into cell and neighborhood tokens
new_token_dict = {
    (key if "ENS" in key else key): value
    for value, key in enumerate(token_dict.keys())}

# Write to disk
with open(nichejepa_token_dictionary_file, "wb") as file:
    pickle.dump(new_token_dict, file)

In [ ]:
# Load vocab
with open(nichejepa_token_dictionary_file, "rb") as file:
    token_dict = pickle.load(file)

token_dict_reverse = {value: key for key, value in token_dict.items()}

In [ ]:
# NicheJEPA token dict
len(token_dict)

In [ ]:
# First n tokens
n = 5
for key, value in list(token_dict.items())[:n]:
    print(f"{key}: {value}")

### 2.2 Compute Normalization Factors for Gene Counts

Corpus means, non-zero medians, or log means are used depending on the 'norm_method' specified in the CellGraphRankTokenizer (check source code).

In [ ]:
# Compute Corpus Means and Non-zero Medians for Cell and Neighborhood Genes
if norm_factors:
    for norm_factor in norm_factors:
        # Load data
        if n_batches == 1:
            adata = sc.read_h5ad(f"{st_data_silver_folder_path}/{dataset_name}/{dataset_name}.h5ad")
            if type(adata.X) == sp._csr.csr_matrix:
                adata.X = adata.X.toarray()
        
            # Filter cells that did not pass QC
            if "filter_pass" in adata.obs.columns:
                filter_pass_idx = np.where([filter_pass == 1 for filter_pass in adata.obs["filter_pass"]])[0]
            else:
                print(f"adata has no column 'filter_pass'; utilizing all cells.")
                filter_pass_idx = np.array([i for i in range(adata.shape[0])])
            adata = adata[filter_pass_idx].copy()
            
            if norm_factor == "read_depth":
                # Normalize cell gene counts by read depth
                adata.X = adata.X / adata.X.sum(1).reshape(-1, 1) * 10000
            elif norm_factor == "cell_area":
                # Normalize cell gene counts by cell area
                adata.X = adata.X / adata.obs["cell_area"].values.reshape(-1, 1)
            elif norm_factor == "prenormalized":
                pass
        else:
            adata_batch_list = []
            for batch_idx in range(1, n_batches+1):
                adata_batch = sc.read_h5ad(f"{st_data_silver_folder_path}/{dataset_name}/{dataset_name}_batch{batch_idx}.h5ad")
                if type(adata_batch.X) == sp._csr.csr_matrix:
                    adata_batch.X = adata_batch.X.toarray()
                
                # Filter cells that did not pass QC
                if "filter_pass" in adata_batch.obs.columns:
                    filter_pass_idx = np.where([filter_pass == 1 for filter_pass in adata_batch.obs["filter_pass"]])[0]
                else:
                    print(f"adata has no column 'filter_pass'; utilizing all cells.")
                    filter_pass_idx = np.array([i for i in range(adata_batch.shape[0])])
                adata_batch = adata_batch[filter_pass_idx].copy()
    
                if norm_factor == "read_depth":
                    # Normalize cell gene counts by read depth
                    adata_batch.X = adata_batch.X / adata_batch.X.sum(1).reshape(-1, 1) * 10000
                elif norm_factor == "cell_area":
                    # Normalize cell gene counts by cell area
                    adata_batch.X = adata_batch.X / adata_batch.obs["cell_area"].values.reshape(-1, 1)
                elif norm_factor == "prenormalized":
                    pass
                        
                adata_batch_list.append(adata_batch)
            adata = ad.concat(adata_batch_list, join="inner", axis=0)
    
            # Combine spatial neighborhood graphs as disconnected components
            batch_connectivities = []
            len_before_batch = 0
            for i in range(len(adata_batch_list)):
                if i == 0: # first batch
                    after_batch_connectivities_extension = sp.csr_matrix(
                        (adata_batch_list[0].shape[0],
                        (adata.shape[0] -
                        adata_batch_list[0].shape[0])))
                    batch_connectivities.append(sp.hstack(
                        (adata_batch_list[0].obsp["spatial_connectivities"],
                        after_batch_connectivities_extension)))
                elif i == (len(adata_batch_list) - 1): # last batch
                    before_batch_connectivities_extension = sp.csr_matrix(
                        (adata_batch_list[i].shape[0],
                        (adata.shape[0] -
                        adata_batch_list[i].shape[0])))
                    batch_connectivities.append(sp.hstack(
                        (before_batch_connectivities_extension,
                        adata_batch_list[i].obsp["spatial_connectivities"])))
                else: # middle batches
                    before_batch_connectivities_extension = sp.csr_matrix(
                        (adata_batch_list[i].shape[0], len_before_batch))
                    after_batch_connectivities_extension = sp.csr_matrix(
                        (adata_batch_list[i].shape[0],
                        (adata.shape[0] -
                        adata_batch_list[i].shape[0] -
                        len_before_batch)))
                    batch_connectivities.append(sp.hstack(
                        (before_batch_connectivities_extension,
                        adata_batch_list[i].obsp["spatial_connectivities"],
                        after_batch_connectivities_extension)))
                len_before_batch += adata_batch_list[i].shape[0]
            adata.obsp["spatial_connectivities"] = sp.vstack(batch_connectivities)
    
            # Copy ensembl IDs
            adata.var = adata_batch_list[0].var
        
        # Compute means and non-zero medians
        gene_means = np.mean(adata.X, axis=0)
        masked_data = np.ma.masked_equal(adata.X, 0)
        gene_nzmedians = np.ma.mean(masked_data, axis=0)
        gene_nzmedians = gene_nzmedians.filled(0)
    
        sc.pp.log1p(adata)
        gene_logmeans = np.mean(adata.X, axis=0)
    
        gene_logmeans = np.array(gene_logmeans).reshape(-1)
        gene_nzmedians = np.array(gene_nzmedians).reshape(-1)
        gene_means = np.array(gene_means).reshape(-1)
     
        # Create norm factor dictionaries for tokenizer and store on disk
        gene_means_dict = {}
        gene_nzmedians_dict = {}
        gene_logmeans_dict = {}
        
        for (gene,
             gene_mean,
             gene_nzmedian,
             gene_logmean,) in zip(
                adata.var["ensembl_id"].values,
                gene_means,
                gene_nzmedians,
                gene_logmeans):
            gene_means_dict[gene] = gene_mean
            gene_nzmedians_dict[gene] = gene_nzmedian
            gene_logmeans_dict[gene] = gene_logmean
        
        with open(gene_means_dictionary_file.replace("gene", f"gene_{norm_factor}"), "wb") as file:
            pickle.dump(gene_means_dict, file)
        with open(gene_nzmedians_dictionary_file.replace("gene", f"gene_{norm_factor}"), "wb") as file:
            pickle.dump(gene_nzmedians_dict, file)
        with open(gene_logmeans_dictionary_file.replace("gene", f"gene_{norm_factor}"), "wb") as file:
            pickle.dump(gene_logmeans_dict, file)

In [ ]:
if norm_factors:
    values = []
    for key, value in gene_means_dict.items():
        values.append(value)
    
    np.mean(values)

### 2.3 Run Tokenizer

In [ ]:
# Tokenize data with different normalization methods
for model_input_size in [len(token_dict) * 2]:
    for norm_method in ["none"]:
       
        output_file_prefix = f"{dataset_name}_{norm_method}_{model_input_size}"

        tk = tokenizers.CellNeighborhoodRankTokenizer(
                model_input_size=model_input_size,
                norm_method=norm_method,
                separate_cell_and_neighborhood_tokens=separate_cell_and_neighborhood_tokens,
                cell_gene_means_file=cell_gene_means_dictionary_file.replace("gene", f"gene_read_depth"),
                cell_gene_nzmedians_file=cell_gene_nzmedians_dictionary_file.replace("gene", f"gene_read_depth"),
                cell_gene_logmeans_file=cell_gene_logmeans_dictionary_file.replace("gene", f"gene_read_depth"),
                neighborhood_gene_means_file=neighborhood_gene_means_dictionary_file.replace("gene", f"gene_read_depth"),
                neighborhood_gene_nzmedians_file=neighborhood_gene_nzmedians_dictionary_file.replace("gene", f"gene_read_depth"),
                neighborhood_gene_logmeans_file=neighborhood_gene_logmeans_dictionary_file.replace("gene", f"gene_read_depth"),
                token_dictionary_file=nichejepa_token_dictionary_file,
                custom_attr_name_dict=custom_attr_name_dict)

        tk.tokenize_data(input_directory=input_directory,
                         output_directory=output_directory,
                         output_file_prefix=output_file_prefix)

### 2.4 Test Tokenizer

In [ ]:
norm_method = "none"
model_input_size = len(token_dict) * 2
tokenized_data_folder_path = f"{sc_data_gold_folder_path}/{tokenizer}/{dataset_name}/{dataset_name}_{norm_method}_{model_input_size}.dataset"
dataset = load_from_disk(tokenized_data_folder_path, keep_in_memory=True)

In [ ]:
dataset

In [ ]:
print(dataset["gene_tokens_cell"][0][:10])
print(dataset["gene_tokens_neighborhood"][0][:10])
print(dataset["cell_types"][0])
print(dataset["input_ids"][0][:10])
print(dataset["input_ids"][0][int(model_input_size/2):int(model_input_size/2)+10])

### 2.5 Evaluate Tokenizer

#### 2.5.1 Cell Classification

In [ ]:
# Load vocab
with open(nichejepa_token_dictionary_file, "rb") as file:
    token_dict = pickle.load(file)

token_dict_reverse = {value: key for key, value in token_dict.items()}

n_gene_tokens = len([token for token in token_dict.keys() if "ENS" in token])

In [ ]:
# Get scgpt gene embeddings for NicheJEPA input token ids
def get_classification_tokens(example, top_n, randomize=False):
    if randomize:
        random.shuffle(example["gene_tokens_cell"])
    example[f"top_{top_n}_cell_tokens"] = [
        token_dict_reverse[token_id].replace("_cell", "")
        for token_id in example["gene_tokens_cell"][:top_n]]           
    return example

In [ ]:
# Test classification token retrieval
top_n = 10
model_input_size = 5562

norm_method = "none"
normalization = f"{norm_method}"
    
output_file_prefix = f"{dataset_name}_{normalization}_{model_input_size}"
tokenized_data_folder_path = f"{sc_data_gold_folder_path}/{tokenizer}/{dataset_name}/{output_file_prefix}.dataset"
dataset = load_from_disk(tokenized_data_folder_path, keep_in_memory=True)
dataset = dataset.map(get_classification_tokens,
                      fn_kwargs={"top_n": top_n},
                      #cache_file_name=dataset_cache_file,
                     )

print(dataset[f"top_{top_n}_cell_tokens"][0])
print("")

top_n = n_gene_tokens
dataset = dataset.map(get_classification_tokens,
                      fn_kwargs={"top_n": top_n},
                      #cache_file_name=dataset_cache_file,
                     )

print(dataset[f"top_{top_n}_cell_tokens"][0])
print("")

##### 2.5.1.1 Baseline (Random Gene Embeddings)

Task: cell type classification using randomly embedded top ranked genes.

In [ ]:
emsemblid_embedding_mapping = {}
for key in token_dict.keys():
   emsemblid_embedding_mapping[key] = np.random.randn(512)

In [ ]:
mode = "nopos"
top_n = n_gene_tokens # n_gene_tokens, 30, 10

randomize = True
if randomize:
    random.seed(42)
    model_input_size = 5562
else:
    if top_n == n_gene_tokens:
        model_input_size = 5562
    else:
        model_input_size = 5562

scores = []

os.makedirs(f"{results_folder_path}/{tokenizer}/{dataset_name}", exist_ok=True)

for norm_method in ["none"]:
    normalization = f"{norm_method}"
        
    output_file_prefix = f"{dataset_name}_{normalization}_{model_input_size}"
    tokenized_data_folder_path = f"{sc_data_gold_folder_path}/{tokenizer}/{dataset_name}/{output_file_prefix}.dataset"
    dataset = load_from_disk(tokenized_data_folder_path, keep_in_memory=True)
    
    dataset = dataset.map(get_classification_tokens,
                          fn_kwargs={"top_n": top_n, "randomize": randomize},
                          #cache_file_name=dataset_cache_file,
                         )

    X_cell = np.array(
        [[emsemblid_embedding_mapping.get(token, np.random.randn(512)) for token in sequence] for
         sequence in dataset[f"top_{top_n}_cell_tokens"]])
    X_cell = np.mean(X_cell, axis=1) # aggregate across tokens
    X_cell = X_cell.reshape(X_cell.shape[0], -1)

    token_type = "cell"

    for label_cat in label_cats:
        label_encoder = LabelEncoder()
        labels = np.array(
            label_encoder.fit_transform(dataset[label_cat]))

        for random_state in range(10):

            np.random.seed(random_state)
        
            indices = np.arange(X_cell.shape[0])
            np.random.shuffle(indices)
            num_train = int(0.8 * len(indices))
            train_indices = indices[:num_train]
            test_indices = indices[num_train:]
            labels_train = labels[train_indices]
            labels_test = labels[test_indices]
            X = X_cell
            X_train = X[train_indices]
            X_test = X[test_indices]
            
            lr_clf = LogisticRegression()
            lr_clf.fit(X_train, labels_train)
            train_predictions = lr_clf.predict(X_train)
            train_f1 = f1_score(labels_train, train_predictions, average="weighted")
            train_score = lr_clf.score(X_train, labels_train)
            test_predictions = lr_clf.predict(X_test)
            test_f1 = f1_score(labels_test, test_predictions, average="weighted")
            test_score = lr_clf.score(X_test, labels_test)
            scores.append({"tokenizer": tokenizer,
                           "normalization": normalization,
                           "token_type": token_type,
                           "clf_task": label_cat,
                           "random_state": random_state,
                           "clf_category": "train",
                           "f1_score": train_f1,
                           "accuracy": train_score})
            scores.append({"tokenizer": tokenizer,
                           "normalization": normalization,
                           "token_type": token_type,
                           "clf_task": label_cat,
                           "random_state": random_state,
                           "clf_category": "test",
                           "f1_score": test_f1,
                           "accuracy": test_score})

            scores_df = pd.DataFrame(scores)
            scores_df.to_csv(f"""{results_folder_path}/{tokenizer}/{dataset_name}/
                             clf_scores_baseline_top{top_n}_{mode}{'_randomized' if randomize else ''}.csv""", index=False)